# Lab: K-Means Clustering

## 1. Học có giám sát vs không giám sát

Đến giờ ta đã làm **học có giám sát** — input có nhãn, model học để dự đoán nhãn cho input mới.

**Học không giám sát** (unsupervised): chỉ có input, không có nhãn. Mục tiêu: *khám phá cấu trúc* trong dữ liệu. Một bài toán điển hình là **clustering** — chia dữ liệu thành các cụm (group) tự nhiên.

Ví dụ ứng dụng:
- Phân khúc khách hàng theo hành vi mua sắm.
- Nhóm bài viết theo chủ đề (không cần biết trước có những chủ đề gì).
- Compress ảnh: quantize 16M màu thành 16 màu chính.

## 2. Ý tưởng K-Means

Cho $K$ (số cụm), K-Means tìm $K$ "tâm" (centroid) sao cho tổng khoảng cách bình phương từ mỗi điểm đến tâm cụm gần nhất là **nhỏ nhất**.

$$
L = \sum_{k=1}^{K} \sum_{x \in C_k} \|x - \mu_k\|^2
$$

Với $\mu_k$ là tâm của cụm $k$, $C_k$ là tập điểm thuộc cụm $k$.

## 3. Thuật toán Lloyd (kinh điển)

1. **Khởi tạo**: chọn $K$ tâm ngẫu nhiên.
2. **Bước Assign**: gán mỗi điểm vào cụm có tâm gần nhất (Euclidean distance).
3. **Bước Update**: cập nhật mỗi tâm = trung bình các điểm trong cụm.
4. Lặp lại 2-3 đến khi tâm không đổi (hoặc đạt max_iter).

Mỗi vòng đảm bảo loss giảm → thuật toán hội tụ. Nhưng **không đảm bảo cực tiểu toàn cục** — chạy lại với khởi tạo khác có thể ra kết quả khác.

## 4. Bốn vấn đề thực tế

### 4.1. Chọn $K$ thế nào?
Hai phương pháp phổ biến:
- **Elbow method**: vẽ inertia (= tổng MSE trong cụm) theo $K$. Chọn $K$ tại "khuỷu tay" — chỗ inertia bắt đầu giảm chậm.
- **Silhouette score**: đo độ "tách" giữa các cụm. Cao = tốt.

### 4.2. Khởi tạo kém → kết quả tệ
Random init có thể đẩy thuật toán vào local minimum. **K-Means++** (mặc định trong sklearn) chọn tâm thông minh hơn → ổn định.

### 4.3. Phải scale feature
K-Means dùng Euclidean distance → feature scale lớn lấn át. Phải `StandardScaler` trước khi clustering.

### 4.4. Chỉ phù hợp với cụm tròn
K-Means giả định cụm dạng *tròn, kích thước tương đồng*. Với cụm cong (như hai trăng lưỡi liềm) hoặc kích cỡ rất khác nhau, K-Means thất bại — dùng **DBSCAN** hoặc **Spectral Clustering** thay thế.

# THỰC HÀNH 1: K-Means trên dữ liệu giả lập 2D

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs, make_moons
from sklearn.cluster import KMeans, DBSCAN
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, silhouette_samples

np.random.seed(42)

# Sinh 4 cụm tròn
X, y_true = make_blobs(n_samples=300, centers=4, cluster_std=0.7, random_state=42)

plt.figure(figsize=(7, 5))
plt.scatter(X[:, 0], X[:, 1], s=20, alpha=0.7)
plt.title('Dữ liệu thô (chưa biết nhãn)'); plt.grid(alpha=0.3); plt.show()

In [ ]:
km = KMeans(n_clusters=4, n_init=10, random_state=42)
labels = km.fit_predict(X)

plt.figure(figsize=(7, 5))
plt.scatter(X[:, 0], X[:, 1], c=labels, cmap='viridis', s=20, alpha=0.7)
plt.scatter(km.cluster_centers_[:, 0], km.cluster_centers_[:, 1],
            c='red', marker='X', s=200, edgecolor='black', linewidth=2,
            label='Centroids')
plt.title(f'K-Means k=4, inertia = {km.inertia_:.2f}')
plt.legend(); plt.grid(alpha=0.3); plt.show()

## 5. Chọn K bằng Elbow + Silhouette

In [ ]:
Ks = list(range(2, 11))
inertias, silhouettes = [], []
for k in Ks:
    km_k = KMeans(n_clusters=k, n_init=10, random_state=42).fit(X)
    inertias.append(km_k.inertia_)
    silhouettes.append(silhouette_score(X, km_k.labels_))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(Ks, inertias, 'o-')
axes[0].set_xlabel('K'); axes[0].set_ylabel('Inertia (within-cluster SSE)')
axes[0].set_title('Elbow method'); axes[0].grid(alpha=0.3)
axes[0].axvline(4, color='red', linestyle='--', alpha=0.5, label='Khuỷu tay tại K=4')
axes[0].legend()

axes[1].plot(Ks, silhouettes, 'o-', color='green')
axes[1].set_xlabel('K'); axes[1].set_ylabel('Silhouette score')
axes[1].set_title('Silhouette (cao = tốt)'); axes[1].grid(alpha=0.3)
best_k = Ks[int(np.argmax(silhouettes))]
axes[1].axvline(best_k, color='red', linestyle='--', alpha=0.5,
                label=f'Best K = {best_k}')
axes[1].legend()
plt.tight_layout(); plt.show()

## 6. Cài K-Means từ scratch

Để hiểu rõ thuật toán, tự cài và so với sklearn.

In [ ]:
class MyKMeans:
    def __init__(self, n_clusters=4, max_iter=100, tol=1e-4, random_state=42):
        self.n_clusters = n_clusters
        self.max_iter = max_iter
        self.tol = tol
        self.random_state = random_state

    def fit(self, X):
        rng = np.random.RandomState(self.random_state)
        # Khởi tạo: chọn n_clusters điểm ngẫu nhiên làm tâm
        idx = rng.choice(len(X), self.n_clusters, replace=False)
        centers = X[idx].copy()

        for it in range(self.max_iter):
            # Assign: mỗi điểm gán vào tâm gần nhất
            d = np.sqrt(((X[:, None, :] - centers[None, :, :]) ** 2).sum(axis=2))
            labels = d.argmin(axis=1)

            # Update: tâm = mean của các điểm trong cụm
            new_centers = np.array([X[labels == k].mean(axis=0)
                                     if (labels == k).any() else centers[k]
                                     for k in range(self.n_clusters)])

            shift = np.linalg.norm(new_centers - centers)
            centers = new_centers
            if shift < self.tol:
                break

        self.cluster_centers_ = centers
        self.labels_ = labels
        self.n_iter_ = it + 1
        self.inertia_ = ((X - centers[labels]) ** 2).sum()
        return self

    def predict(self, X):
        d = np.sqrt(((X[:, None, :] - self.cluster_centers_[None, :, :]) ** 2).sum(axis=2))
        return d.argmin(axis=1)

mine = MyKMeans(n_clusters=4).fit(X)
print(f'My KMeans   inertia: {mine.inertia_:.2f}, hội tụ sau {mine.n_iter_} iter')
print(f'sklearn     inertia: {km.inertia_:.2f}')

## 7. Khi K-Means thất bại — cụm cong

Hai trăng lưỡi liềm — K-Means giả định cụm tròn nên không xử lý được.

In [ ]:
X_moon, y_moon = make_moons(n_samples=300, noise=0.08, random_state=42)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# K-Means
km_moon = KMeans(n_clusters=2, n_init=10, random_state=42).fit(X_moon)
axes[0].scatter(X_moon[:, 0], X_moon[:, 1], c=km_moon.labels_, cmap='coolwarm', s=20)
axes[0].scatter(km_moon.cluster_centers_[:, 0], km_moon.cluster_centers_[:, 1],
                c='black', marker='X', s=200)
axes[0].set_title('K-Means: SAI — cắt ngang hai trăng')
axes[0].grid(alpha=0.3)

# DBSCAN — phát hiện cụm theo mật độ
dbscan = DBSCAN(eps=0.2, min_samples=5).fit(X_moon)
axes[1].scatter(X_moon[:, 0], X_moon[:, 1], c=dbscan.labels_, cmap='coolwarm', s=20)
axes[1].set_title(f'DBSCAN: ĐÚNG — bám theo mật độ ({len(set(dbscan.labels_))} cụm)')
axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

print('K-Means không xử lý được cụm cong. DBSCAN, Spectral Clustering giải quyết tốt hơn.')

## 8. Ứng dụng thực tế: nén ảnh bằng K-Means

Một ảnh màu thường có 16M màu. Nếu chỉ giữ $K=16$ màu chính (bằng cách clustering pixel), ảnh nhỏ hơn 1000 lần mà vẫn nhận diện được.

In [ ]:
from sklearn.datasets import load_sample_image

# Tải ảnh sample (đã có trong sklearn)
try:
    img = load_sample_image('china.jpg')
except Exception:
    print('Không tải được ảnh sample, bỏ qua phần này')
    img = None

if img is not None:
    h, w, c = img.shape
    pixels = img.reshape(-1, 3) / 255.0

    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    axes[0].imshow(img); axes[0].set_title('Gốc (16M màu)'); axes[0].axis('off')

    for ax, K in zip(axes[1:], [4, 8, 16]):
        # Lấy mẫu để fit nhanh
        sample = pixels[np.random.choice(len(pixels), 5000, replace=False)]
        km = KMeans(n_clusters=K, n_init=3, random_state=42).fit(sample)
        new_pixels = km.cluster_centers_[km.predict(pixels)]
        new_img = new_pixels.reshape(h, w, 3)
        ax.imshow(new_img); ax.set_title(f'K = {K} màu'); ax.axis('off')
    plt.tight_layout(); plt.show()

## Tổng kết

1. K-Means: thuật toán **không giám sát**, chia dữ liệu thành $K$ cụm sao cho tổng khoảng cách trong cụm nhỏ nhất.
2. Lloyd's algorithm: lặp Assign → Update đến hội tụ.
3. **Chọn K**: Elbow method và Silhouette score.
4. **K-Means++** (default sklearn) khởi tạo thông minh hơn random.
5. **Phải scale feature** vì dùng Euclidean distance.
6. K-Means *thất bại* với cụm cong → DBSCAN, Spectral Clustering.
7. Ứng dụng: customer segmentation, image quantization, document clustering.

# BÀI TẬP VỀ NHÀ

## Bài 1: Customer segmentation
Tự sinh dataset 500 "khách hàng" với 3 feature: tuổi, thu nhập, số lần mua/tháng. Apply K-Means với K=3,4,5. Dùng silhouette để chọn K tốt nhất. Mô tả từng cụm: tuổi/thu nhập/mua sắm có gì đặc trưng?

*Gợi ý:* `np.random.normal` để sinh, scale trước khi cluster, đặt tên cho cụm như "trẻ–chi nhiều", "trung niên–chi vừa"...

## Bài 2: Ảnh hưởng của khởi tạo
Trên `make_blobs` 4 cụm:
1. Train K-Means với `init='random'`, lặp 10 lần với 10 random_state khác nhau.
2. Train với `init='k-means++'`, cũng 10 lần.
3. Vẽ histogram của inertia. K-means++ có ổn định hơn không?

## Bài 3: Quên scale
Lấy dataset Iris, chia làm 2 phiên bản:
- Bản A: scale chuẩn (StandardScaler).
- Bản B: KHÔNG scale.

Train K-Means k=3. So sánh kết quả với nhãn thật (dùng `adjusted_rand_score`). Bản nào tốt hơn? Vì sao?

## Bài 4: Image quantization tự chọn ảnh
Dùng ảnh bất kỳ (load bằng `PIL.Image` hoặc `cv2.imread`). Apply K-Means với K = 2, 4, 8, 16, 32, 64 màu. Lưu các phiên bản và so sánh. Tại K nào ảnh "đủ giống" gốc?

## Bài 5: K-Means++ from scratch
Mở rộng class `MyKMeans` ở trên: thay khởi tạo random bằng K-Means++.

Thuật toán:
1. Chọn ngẫu nhiên 1 điểm làm tâm đầu tiên.
2. Với mỗi điểm còn lại, tính khoảng cách $D(x)$ đến tâm gần nhất hiện có.
3. Chọn điểm tiếp theo với xác suất tỷ lệ với $D(x)^2$ — điểm xa các tâm đã có thì khả năng được chọn cao.
4. Lặp đến đủ K tâm.

So sánh inertia trung bình (qua 10 seed) của random init vs K-Means++ init.